# Comparison of predicted/measured thermal effects across TUS studies

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd

from usnm2p.logger import logger
from usnm2p.constants import *
from usnm2p.utils import compute_radiation_force, compute_temperature_increase
from usnm2p.fileops import get_data_root
from usnm2p.plotters import *

### Load data 

In [16]:
fname = 'TUS_studies_temp_data.xlsx'
fpath = os.path.join(get_data_root(), fname)
data = pd.read_excel(fpath).set_index('Parameter')
stimparams = [k for k in data.index if k != 'measured ΔT (°C)']

,Murphy 2022,our study
Parameter,,
f_0 (MHz),0.623,2.10
PRF (Hz),900.000,100.00
I_SPPA (W/cm2),4.500,19.80
DC (%),20.000,80.00
duration (s),5.000,0.20
measured ΔT (°C),0.230,0.94


### Compute estimated radiation force and temperature rise for each study

In [23]:
# Compute ISPTA from ISPPA and duty cycle
data.loc['I_SPTA (W/cm2)'] = data.loc['I_SPPA (W/cm2)'] * data.loc['DC (%)'] / 100

# Compute radiation force, heat generation rate, and temperature increase
data.loc['Frad (N/m3)'] = compute_radiation_force(
    data.loc['f_0 (MHz)'],
    data.loc['I_SPPA (W/cm2)']
)
data.loc['predicted Qheat (°C/s)'] = compute_heat_generation_rate(
    data.loc['f_0 (MHz)'],
    data.loc['I_SPTA (W/cm2)'],
)
data.loc['predicted ΔT (°C)'] = compute_temperature_increase(
    data.loc['f_0 (MHz)'],
    data.loc['I_SPTA (W/cm2)'],
    data.loc['duration (s)'],
)

data.loc['ratio predicted / measured ΔT'] = data.loc['predicted ΔT (°C)'] / data.loc['measured ΔT (°C)']

data

,Murphy 2022,our study,our study (in water)
Parameter,,,
f_0 (MHz),0.623000,2.100000,2.100000
PRF (Hz),900.000000,100.000000,100.000000
I_SPPA (W/cm2),4.500000,19.800000,19.800000
DC (%),20.000000,80.000000,80.000000
duration (s),5.000000,0.200000,0.200000
measured ΔT (°C),0.230000,0.940000,NaN
I_SPTA (W/cm2),0.900000,15.840000,15.840000
Frad (N/m3),214.040744,4570.879283,4570.879283
predicted Qheat (°C/s),0.017433,1.489173,1.489173


In [27]:
# Add "our study (in water)" to the data table
k = 'our study (in water)'
data[k] = data.loc[[*stimparams, 'I_SPTA (W/cm2)'], 'our study'].copy()

# Compute radiation force, heat generation rate, and temperature increase
data.loc['Frad (N/m3)', k] = compute_radiation_force(
    data.loc['f_0 (MHz)', k],
    data.loc['I_SPPA (W/cm2)', k],
    c=C_WATER, alpha0=ALPHA0_WATER, b=B_WATER
)
data.loc['predicted Qheat (°C/s)', k] = compute_heat_generation_rate(
    data.loc['f_0 (MHz)', k],
    data.loc['I_SPTA (W/cm2)', k],
    rho=RHO_WATER, C=CS_WATER,
    alpha0=ALPHA0_WATER, b=B_WATER, 
)

data.loc['predicted ΔT (°C)', k] = compute_temperature_increase(
    data.loc['f_0 (MHz)', k],
    data.loc['I_SPTA (W/cm2)', k],
    data.loc['duration (s)', k],
    rho=RHO_WATER, C=CS_WATER,
    alpha0=ALPHA0_WATER, b=B_WATER, 
)

data

,Murphy 2022,our study,our study (in water)
Parameter,,,
f_0 (MHz),0.623000,2.100000,2.100000
PRF (Hz),900.000000,100.000000,100.000000
I_SPPA (W/cm2),4.500000,19.800000,19.800000
DC (%),20.000000,80.000000,80.000000
duration (s),5.000000,0.200000,0.200000
measured ΔT (°C),0.230000,0.940000,NaN
I_SPTA (W/cm2),0.900000,15.840000,15.840000
Frad (N/m3),214.040744,4570.879283,14.209760
predicted Qheat (°C/s),0.017433,1.489173,0.004057
